In [1]:
import os
# os.system("ros2 pkg list")
# os.system("source /workspace/stretch_ros2/install/setup.sh")

In [7]:
os.system("ros2 node list")

/joint_state_publisher
/robot_state_publisher
/rviz
/stretch_sim_driver
/transform_listener_impl_5578f76a9f00


0

In [8]:
os.system("ros2 topic list")

/camera/d405/color
/camera/d405/depth
/camera/d435i/color
/camera/d435i/depth
/camera/nav/color
/clicked_point
/goal_pose
/initialpose
/joint_pose_cmd
/joint_states
/move_by_pos_cmd
/odom
/parameter_events
/robot_description
/rosout
/stretch/cmd_vel
/stretch/joint_states
/tf
/tf_static


0

In [9]:
os.system("ros2 service list")

/home_the_robot
/joint_state_publisher/describe_parameters
/joint_state_publisher/get_parameter_types
/joint_state_publisher/get_parameters
/joint_state_publisher/list_parameters
/joint_state_publisher/set_parameters
/joint_state_publisher/set_parameters_atomically
/robot_state_publisher/describe_parameters
/robot_state_publisher/get_parameter_types
/robot_state_publisher/get_parameters
/robot_state_publisher/list_parameters
/robot_state_publisher/set_parameters
/robot_state_publisher/set_parameters_atomically
/rviz/describe_parameters
/rviz/get_parameter_types
/rviz/get_parameters
/rviz/list_parameters
/rviz/set_parameters
/rviz/set_parameters_atomically
/stop_the_robot
/stow_the_robot
/stretch_sim_driver/describe_parameters
/stretch_sim_driver/get_parameter_types
/stretch_sim_driver/get_parameters
/stretch_sim_driver/list_parameters
/stretch_sim_driver/set_parameters
/stretch_sim_driver/set_parameters_atomically


0

In [ ]:
# Stop the robot
os.system('ros2 service call /stop_the_robot std_srvs/srv/Trigger "{}"')

In [11]:
# Home the robot
os.system('ros2 service call /home_the_robot std_srvs/srv/Trigger "{}"')

waiting for service to become available...
requester: making request: std_srvs.srv.Trigger_Request()

response:
std_srvs.srv.Trigger_Response(success=True, message='Homed the robot.')



0

In [42]:

# Stow the robot
os.system('ros2 service call /stow_the_robot std_srvs/srv/Trigger "{}"')

requester: making request: std_srvs.srv.Trigger_Request()

response:
std_srvs.srv.Trigger_Response(success=True, message='Stowed the robot.')



0

In [ ]:
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Twist
from std_msgs.msg import Float64MultiArray
import stretch_mujoco.config as config
# import numpy as np



try:
    rclpy.init()
except:
    print("rclpy already initialized")

In [ ]:
class StretchMujocoPublisher(Node):
    def __init__(self):
        super().__init__('mujoco_publisher')

        self.base_vel_pub = self.create_publisher(Twist, 'cmd_vel', 10)
        self.joint_pose_pub = self.create_publisher(Float64MultiArray, 'joint_pose_cmd', 10)
        self.move_by_pos_pub = self.create_publisher(Float64MultiArray, 'move_by_pos_cmd', 10)

    def publish_cmd_vel(self, linear_x=0.0, angular_z=0.0):
        """ Publish Twist message to control velocity """
        msg = Twist()
        msg.linear.x = linear_x
        msg.angular.z = angular_z
        self.base_vel_pub.publish(msg)
        self.get_logger().info(f'Published cmd_vel: linear_x={linear_x}, angular_z={angular_z}')

    def publish_joint_pose(self, positions):
        """ Publish Float64MultiArray for joint positions """
        msg = Float64MultiArray()
        msg.data = positions
        self.joint_pose_pub.publish(msg)
        self.get_logger().info(f'Published joint_pose_cmd: {positions}')

    def publish_move_by_pos(self, positions):
        """ Publish Float64MultiArray for move-by-position command """
        msg = Float64MultiArray()
        msg.data = positions
        self.move_by_pos_pub.publish(msg)
        self.get_logger().info(f'Published move_by_pos_cmd: {positions}')

node = StretchMujocoPublisher()


In [43]:
print("Move_to and move_by msg.data in sequence of allowed position actuators:")
allowed_actuators = config.allowed_position_actuators
print("config allowed actuators:",allowed_actuators)
mujoco_actuators = ["left_wheel_vel", "right_wheel_vel", "lift", "arm", "wrist_yaw", "wrist_pitch", "wrist_roll",  "gripper", "head_pan", "head_tilt",]
print("mujoco xml actuators:", mujoco_actuators)

idx_mujoco2config ={
    0: None, 1: None, 2: 4, 3: 0, 4: 7, 5: 5, 6: 6, 7: 1, 8: 2, 9: 3
}

idx_config2mujoco = dict()
for key, value in idx_mujoco2config.items():
    if value is not None:
        idx_config2mujoco[value] = key

def to_float_list(list):
    return [float(x) for x in list]

# Test dict
# for i in range(10):
#     if i in idx_config2mujoco:
#         print("config:", allowed_actuators[i], "mujoco:", mujoco_actuators[idx_config2mujoco[i]])

Move_to and move_by msg.data in sequence of allowed position actuators:
config allowed actuators: ['arm', 'gripper', 'head_pan', 'head_tilt', 'lift', 'wrist_pitch', 'wrist_roll', 'wrist_yaw', 'base_rotate', 'base_translate']
mujoco xml actuators: ['left_wheel_vel', 'right_wheel_vel', 'lift', 'arm', 'wrist_yaw', 'wrist_pitch', 'wrist_roll', 'gripper', 'head_pan', 'head_tilt']


In [41]:
node.publish_cmd_vel(0.0, 0.0)

[INFO] [1740726304.199144183] [mujoco_publisher]: Published cmd_vel: linear_x=0.0, angular_z=0.0


In [45]:
# <key name="home" ctrl="0 0 0.6 0.1 0 0 0 0 0 0"/>
# <key name="stow" ctrl="0 0 0.23 0 3.14 -0.4 0 0 0 0"/>
mujoco_home_qpos = [0.0, 0.0, 0.6, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] 
mujoco_stow_qpos = [0.0, 0.0, 0.23, 0.0, 3.14, -0.4, 0.0, 0.0, 0.0, 0.0] 
config_home_qpos = [mujoco_home_qpos[idx_config2mujoco[i]] for i in range(len(idx_config2mujoco))]
config_stow_qpos = [mujoco_stow_qpos[idx_config2mujoco[i]] for i in range(len(idx_config2mujoco))]

qpos = config_home_qpos
qpos = config_stow_qpos
qpos.extend([0.0, 0.0]) # 'base_rotate', 'base_translate' are invalid and so 0.0

# qpos = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

allowed_actuators = config.allowed_position_actuators
assert len(qpos) == len(allowed_actuators), f"Length of dpos {len(qpos)} does not match allowed_actuators {len(allowed_actuators)}"
node.publish_joint_pose(qpos)

[INFO] [1740726333.027457796] [mujoco_publisher]: Published joint_pose_cmd: [0.0, 0.0, 0.0, 0.0, 0.23, -0.4, 0.0, 3.14, 0.0, 0.0]


In [ ]:
# delta pos
import math
pi = math.pi
# NOTE: We only actuate the first non-zero actuator in the order of allowed_position_actuators
arm             = 0
gripper         = 0
head_pan        = 0
head_tilt       = 0
lift            = 0
wrist_pitch     = 0
wrist_roll      = 0
wrist_yaw       = 0
base_rotate     = pi/2
base_translate  = 1

dpos = [arm, gripper, head_pan, head_tilt, lift, wrist_pitch, wrist_roll, wrist_yaw, base_rotate, base_translate]    # motion following the order of allowed_position_actuators
dpos = to_float_list(dpos)
allowed_actuators = config.allowed_position_actuators
assert len(dpos) == len(allowed_actuators), f"Length of dpos {len(dpos)} does not match allowed_actuators {len(allowed_actuators)}"
node.publish_move_by_pos(dpos)

[INFO] [1740726602.707514661] [mujoco_publisher]: Published move_by_pos_cmd: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.5707963267948966, 1.0]


In [ ]:
node.destroy_node()
rclpy.shutdown()

In [40]:
0.0 == 0.0

True